# HERD v9 — Deep Escape: Loss-Barrier Criterion + Adaptive Step Size

## v8 Post-Mortem: Why Escape Was Too Shallow
v8 confirmed two critical things:
1. **Lanczos eigenvector extraction WORKS** — all 5 seeds found genuine negative curvature (λ_min from -12 to -55)
2. **VGG-16-BN landscape has real structure** — |λ_min/λ_max| ≈ 0.16-0.32 (much stronger than ResNet's 0.15)

But v8's escape was **too shallow**:
- Seeds 42, 123, 777 escaped in **1 step** with fall_distance = **0.05**
- The cosine threshold (0.3) was hit trivially — gradient noise alone flips the cosine
- The model never actually left its basin; it just jittered within the same neighborhood

## What v9 Fixes

### 1. Loss-Barrier Escape Criterion
Instead of cosine similarity (unreliable with stochastic gradients), detect actual **basin crossings**:
- Walk along negative eigenvector
- Track loss: if loss goes UP (climbing the saddle ridge) then comes DOWN → crossed a barrier
- Require a **minimum distance** before allowing escape (avoid false triggers)

### 2. Adaptive Step Size Scaled by Curvature
Step size proportional to 1/√|λ_min| — stronger curvature = smaller steps needed.
This ensures we track the curvature direction faithfully.

### 3. Multi-Direction Escape
Try ALL negative eigenvectors (not just the most negative).
Pick the one that achieves the deepest loss drop after the barrier.

## Hypotheses
- **H1**: Loss-barrier escape detects genuine basin crossings (loss ↑ then ↓)
- **H2**: Escape distance is significantly larger than v8 (>1.0 vs v8's 0.05-0.4)
- **H3**: Post-escape basin has different loss landscape (different spectrum)
- **H4**: HERD v4 achieves higher test accuracy than AdamW (p < 0.05)
- **H5**: HERD v4 finds flatter minima (lower sharpness) than AdamW

In [1]:
# ══════════════════════════════════════════════════════════════
# CELL 1: SETUP
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import copy, time, json, os
from scipy import stats

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB
PyTorch: 2.8.0+cu126


In [2]:
# ══════════════════════════════════════════════════════════════
# CELL 2: DATA + VGG MODEL + UTILITIES
# (Same as v8 — VGG-16-BN for CIFAR-10)
# ══════════════════════════════════════════════════════════════

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
    T.RandomErasing(p=0.1),
])
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])

data_root = '/kaggle/input/cifar10-python' if os.path.isdir('/kaggle/input/cifar10-python') else './data'
train_set = torchvision.datasets.CIFAR10(data_root, train=True, download=True, transform=train_transform)
test_set  = torchvision.datasets.CIFAR10(data_root, train=False, download=True, transform=test_transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,
                                           num_workers=2, pin_memory=True, drop_last=True)
test_loader  = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False,
                                           num_workers=2, pin_memory=True)

print(f"Train: {len(train_set)}, Test: {len(test_set)}")

# ── CutMix ────────────────────────────────────────────────────
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    b = x.size(0)
    idx = torch.randperm(b, device=x.device)
    y_a, y_b = y, y[idx]
    W, H = x.size(2), x.size(3)
    cut_rat = np.sqrt(1 - lam)
    cw, ch = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cw // 2, 0, W)
    y1 = np.clip(cy - ch // 2, 0, H)
    x2 = np.clip(cx + cw // 2, 0, W)
    y2 = np.clip(cy + ch // 2, 0, H)
    x_mixed = x.clone()
    x_mixed[:, :, x1:x2, y1:y2] = x[idx, :, x1:x2, y1:y2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return x_mixed, y_a, y_b, lam

def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ── VGG-16-BN for CIFAR-10 ───────────────────────────────────
class VGG16BN_CIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

def make_vgg16bn():
    model = VGG16BN_CIFAR(num_classes=10)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"VGG-16-BN (CIFAR): {n_params:,} parameters")
    return model

# ── Evaluation + Utilities ────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    total_loss = 0.0
    loss_fn = nn.CrossEntropyLoss()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        total_loss += loss_fn(out, y).item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total, total_loss / total

def get_param_vector(model):
    return torch.cat([p.data.view(-1) for p in model.parameters() if p.requires_grad])

def get_gradient_direction(model, loss_fn, loader, device, num_batches=3):
    model.train()
    model.zero_grad()
    count = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        loss.backward()
        count += 1
        if count >= num_batches:
            break
    grad = torch.cat([p.grad.view(-1) for p in model.parameters() if p.requires_grad and p.grad is not None])
    norm = grad.norm()
    return grad / (norm + 1e-10), norm.item()

def gradient_cosine_similarity(g1, g2):
    return (torch.dot(g1, g2) / (g1.norm() * g2.norm() + 1e-10)).item()

def compute_train_loss(model, loss_fn, loader, device, num_batches=5):
    """Compute train loss on a few batches (deterministic, no augmentation noise)."""
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            total_loss += loss_fn(model(x), y).item()
            count += 1
            if count >= num_batches:
                break
    return total_loss / count

# Quick sanity check
m = make_vgg16bn().to(device)
dummy = torch.randn(2, 3, 32, 32, device=device)
out = m(dummy)
print(f"Output shape: {out.shape}  (expected: [2, 10])")
del m, dummy, out
torch.cuda.empty_cache() if device.type == 'cuda' else None
print("✅ VGG-16-BN + data pipeline ready")

100%|██████████| 170M/170M [00:01<00:00, 106MB/s]


Train: 50000, Test: 10000
VGG-16-BN (CIFAR): 15,253,578 parameters
Output shape: torch.Size([2, 10])  (expected: [2, 10])
✅ VGG-16-BN + data pipeline ready


In [3]:
# ══════════════════════════════════════════════════════════════
# CELL 3: HESSIAN TOOLS — LANCZOS + MULTI-DIRECTION EXTRACTION
#
# Same proven Lanczos from v8 + new: extract ALL negative
# eigenvectors, not just the most negative one.
# ══════════════════════════════════════════════════════════════

def hessian_vector_product(model, loss_fn, loader, device, v, num_batches=5):
    """Compute Hv via autodiff."""
    params = [p for p in model.parameters() if p.requires_grad]
    model.zero_grad()
    total_loss = 0.0
    count = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        total_loss += loss
        count += 1
        if count >= num_batches:
            break
    total_loss = total_loss / count

    grads = torch.autograd.grad(total_loss, params, create_graph=True)
    flat_grad = torch.cat([g.contiguous().view(-1) for g in grads])
    flat_v = torch.cat([vi.contiguous().view(-1) for vi in v])
    grad_v = torch.dot(flat_grad, flat_v)
    hvp = torch.autograd.grad(grad_v, params, retain_graph=False)
    return [h.detach() for h in hvp]


def lanczos_decomposition(model, loss_fn, loader, device, k=20, num_batches=5):
    """Full Lanczos: eigenvalues + eigenvector coefficients + basis vectors."""
    params = [p for p in model.parameters() if p.requires_grad]

    q = [torch.randn_like(p) for p in params]
    norm = torch.sqrt(sum(torch.sum(x**2) for x in q))
    q = [x / (norm + 1e-10) for x in q]

    Q_list = [q]
    alphas = []
    betas = []
    q_prev = [torch.zeros_like(p) for p in params]
    beta_prev = 0.0

    for j in range(k):
        Hq = hessian_vector_product(model, loss_fn, loader, device, q, num_batches)
        alpha = sum(torch.sum(qi * hqi) for qi, hqi in zip(q, Hq)).item()
        alphas.append(alpha)

        r = [hqi - alpha * qi - beta_prev * qpi
             for hqi, qi, qpi in zip(Hq, q, q_prev)]

        for Q_old in Q_list:
            dot = sum(torch.sum(ri * qo) for ri, qo in zip(r, Q_old))
            r = [ri - dot.item() * qo for ri, qo in zip(r, Q_old)]

        beta = torch.sqrt(sum(torch.sum(ri**2) for ri in r)).item()

        if beta < 1e-10:
            break

        if j < k - 1:
            betas.append(beta)
            q_prev = q
            beta_prev = beta
            q = [ri / (beta + 1e-10) for ri in r]
            Q_list.append(q)

    m = len(alphas)
    T = np.zeros((m, m))
    for i in range(m):
        T[i, i] = alphas[i]
    for i in range(len(betas)):
        T[i, i+1] = betas[i]
        T[i+1, i] = betas[i]

    eigenvalues_raw, eigvecs_raw = np.linalg.eigh(T)
    sort_idx = np.argsort(eigenvalues_raw)[::-1]
    eigenvalues = eigenvalues_raw[sort_idx]
    eigvec_coeffs = eigvecs_raw[:, sort_idx]

    spectrum_info = {
        'eigenvalues': eigenvalues.tolist(),
        'n_positive': int(np.sum(eigenvalues > 0.1)),
        'n_negative': int(np.sum(eigenvalues < -0.1)),
        'n_near_zero': int(np.sum(np.abs(eigenvalues) <= 0.1)),
        'max_eigenvalue': float(eigenvalues[0]),
        'min_eigenvalue': float(eigenvalues[-1]),
        'trace': float(np.sum(eigenvalues)),
        'condition': float(abs(eigenvalues[0]) / (abs(eigenvalues[-1]) + 1e-10)),
    }

    return eigenvalues, eigvec_coeffs, Q_list, spectrum_info


def extract_all_negative_eigenvectors(eigenvalues, eigvec_coeffs, Q_list, device,
                                       threshold=-0.1):
    """
    Extract ALL eigenvectors with eigenvalue < threshold from Lanczos decomposition.
    Returns list of (eigenvector, eigenvalue) tuples, sorted by eigenvalue (most negative first).
    """
    neg_eigvecs = []
    m = len(eigenvalues)
    n_basis = len(Q_list)

    for idx in range(m):
        eig_val = eigenvalues[idx]
        if eig_val >= threshold:
            continue

        # Reconstruct eigenvector from Lanczos basis
        coeffs = eigvec_coeffs[:, idx]
        n_use = min(len(coeffs), n_basis)

        v = [torch.zeros_like(Q_list[0][p_idx]) for p_idx in range(len(Q_list[0]))]
        for j in range(n_use):
            c = float(coeffs[j])
            for p_idx in range(len(v)):
                v[p_idx] = v[p_idx] + c * Q_list[j][p_idx]

        # Normalize
        norm = torch.sqrt(sum(torch.sum(x**2) for x in v))
        v = [x / (norm + 1e-10) for x in v]

        neg_eigvecs.append((v, float(eig_val)))

    # Sort: most negative first
    neg_eigvecs.sort(key=lambda x: x[1])

    return neg_eigvecs


print("Hessian tools ready:")
print("  lanczos_decomposition() → full decomposition")
print("  extract_all_negative_eigenvectors() → ALL negative directions")

Hessian tools ready:
  lanczos_decomposition() → full decomposition
  extract_all_negative_eigenvectors() → ALL negative directions


In [4]:
# ══════════════════════════════════════════════════════════════
# CELL 4: HERD v4 — DEEP ESCAPE ENGINE
#
# THE KEY CHANGE FROM v8:
#   v8: cosine threshold → triggered by gradient noise after 1 step
#   v9: loss-barrier detection → requires actual basin crossing
#
# Basin crossing signature:
#   loss goes UP (climbing saddle ridge) then comes DOWN (descending
#   into new basin). We track the loss trajectory and detect this
#   pattern. Additionally, require minimum parameter distance.
#
# Multi-direction: try up to 3 most negative eigenvectors,
# pick the direction that achieves the best post-escape loss.
# ══════════════════════════════════════════════════════════════

TOTAL_EPOCHS = 100
CUTMIX_PROB = 0.5
LR = 0.001
WD = 5e-4
LABEL_SMOOTH = 0.1
SEEDS = [42, 123, 777, 2024, 31415]

# v4 escape config
V4_WARMUP = 10
V4_LANCZOS_K = 20
V4_MAX_ESCAPE_STEPS = 100     # more steps to find barrier
V4_MIN_DISTANCE = 0.5         # must move at least this far before claiming escape
V4_MAX_DIRECTIONS = 3         # try top-3 most negative eigenvectors
V4_LOSS_PATIENCE = 5          # loss must be below start for 5 consecutive steps after barrier

# v8 cached baselines (from v8 Kaggle run)
CACHED_ADAMW_V8 = {
    42:    {'best_acc': 91.35, 'sharpness': 35.11},
    123:   {'best_acc': 91.68, 'sharpness': 24.06},
    777:   {'best_acc': 92.08, 'sharpness': 38.52},
    2024:  {'best_acc': 92.18, 'sharpness': 23.25},
    31415: {'best_acc': 91.63, 'sharpness': 28.48},
}


def train_epochs(model, n_epochs, tag="", verbose=True, save_checkpoints_at=None):
    """Train with optional checkpoint saving."""
    loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)

    best_acc = 0.0
    best_state = None
    checkpoints = {}

    for epoch in range(n_epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            if np.random.rand() < CUTMIX_PROB:
                x, y_a, y_b, lam = cutmix_data(x, y)
                opt.zero_grad()
                loss = mixed_criterion(loss_fn, model(x), y_a, y_b, lam)
            else:
                opt.zero_grad()
                loss = loss_fn(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        actual_epoch = epoch + 1
        if save_checkpoints_at and actual_epoch in save_checkpoints_at:
            checkpoints[actual_epoch] = copy.deepcopy(model.state_dict())

        if actual_epoch % 10 == 0 or epoch == n_epochs - 1:
            test_acc, _ = evaluate(model, test_loader, device)
            if test_acc > best_acc:
                best_acc = test_acc
                best_state = copy.deepcopy(model.state_dict())
            if verbose:
                print(f"    [{tag}] Ep {actual_epoch:3d}/{n_epochs} | "
                      f"TestAcc: {test_acc:.2f}% | Best: {best_acc:.2f}%")

    return {'best_acc': best_acc, 'best_state': best_state, 'checkpoints': checkpoints}


def escape_along_direction(model, direction, eigenvalue, loss_fn, loader, device,
                            max_steps=100, min_distance=0.5, loss_patience=5):
    """
    Walk along a single negative curvature direction using LOSS-BARRIER detection.

    Escape criterion (all must be met):
    1. Loss went UP from start (crossed barrier/ridge)
    2. Loss came back DOWN below start + tolerance (entered new basin)
    3. Parameter distance > min_distance (actually moved)
    4. Loss stays below start for 'loss_patience' consecutive steps

    Step size is adaptive: lr = 1.0 / sqrt(|eigenvalue|), clamped to [0.01, 0.5].
    This scales with curvature strength.
    """
    params = [p for p in model.parameters() if p.requires_grad]
    pre_params = get_param_vector(model)

    # Adaptive step size: stronger curvature → smaller, more precise steps
    escape_lr = 1.0 / np.sqrt(abs(eigenvalue) + 1e-10)
    escape_lr = np.clip(escape_lr, 0.01, 0.5)

    # Get starting loss (eval mode, deterministic)
    start_loss = compute_train_loss(model, loss_fn, loader, device, num_batches=10)

    log = {
        'eigenvalue': eigenvalue,
        'escape_lr': escape_lr,
        'start_loss': start_loss,
        'steps': [],
        'barrier_crossed': False,
        'escaped': False,
        'max_loss_seen': start_loss,
        'barrier_height': 0.0,
    }

    crossed_barrier = False
    below_start_count = 0
    max_loss = start_loss

    for step in range(max_steps):
        # Step along negative eigenvector
        with torch.no_grad():
            for p, vi in zip(params, direction):
                p.add_(vi, alpha=escape_lr)

        # Measure loss (deterministic eval)
        current_loss = compute_train_loss(model, loss_fn, loader, device, num_batches=10)
        dist = torch.norm(get_param_vector(model) - pre_params).item()

        log['steps'].append({
            'step': step + 1,
            'loss': current_loss,
            'distance': dist,
        })

        # Track max loss (barrier height)
        if current_loss > max_loss:
            max_loss = current_loss

        # Check: did we cross a barrier? (loss went above start)
        if current_loss > start_loss + 0.01:  # small tolerance for noise
            crossed_barrier = True
            log['barrier_crossed'] = True

        # Check: are we in a new basin? (loss came back below start after barrier)
        if crossed_barrier and current_loss < start_loss:
            below_start_count += 1
        else:
            below_start_count = 0

        # Escape conditions met?
        if (crossed_barrier and
            below_start_count >= loss_patience and
            dist >= min_distance):
            log['escaped'] = True
            log['reason'] = 'loss_barrier_crossed'
            log['n_steps'] = step + 1
            log['barrier_height'] = max_loss - start_loss
            log['final_loss'] = current_loss
            break

        # Also escape if we've moved far enough and loss is lower (monotone descent = downhill saddle)
        if (dist >= min_distance * 2 and
            current_loss < start_loss - 0.01 and
            not crossed_barrier):
            log['escaped'] = True
            log['reason'] = 'monotone_descent_escape'
            log['n_steps'] = step + 1
            log['barrier_height'] = 0.0
            log['final_loss'] = current_loss
            break

        # Print progress every 20 steps
        if (step + 1) % 20 == 0:
            print(f"          Step {step+1}: loss={current_loss:.4f} "
                  f"(start={start_loss:.4f}, max={max_loss:.4f}), "
                  f"dist={dist:.4f}, barrier={'✅' if crossed_barrier else '❌'}, "
                  f"below_start={below_start_count}")

    log['fall_distance'] = torch.norm(get_param_vector(model) - pre_params).item()
    log['max_loss_seen'] = max_loss

    if 'n_steps' not in log:
        log['n_steps'] = max_steps
        log['reason'] = 'max_steps'
        log['final_loss'] = current_loss
        # Even if max_steps reached, check if we at least crossed a barrier
        if crossed_barrier:
            log['reason'] = 'max_steps_barrier_crossed_but_not_settled'

    return log


def deep_escape_v4(model, loss_fn, loader, device, lanczos_k=20,
                    max_steps=100, min_distance=0.5, max_directions=3,
                    loss_patience=5):
    """
    HERD v4: Multi-direction deep escape with loss-barrier detection.

    1. Compute Lanczos decomposition
    2. Extract ALL negative eigenvectors
    3. Try each direction (up to max_directions)
    4. Pick the direction that achieves the best escape (lowest post-escape loss)
    5. If no escape found, fall back to best partial result
    """
    saved_state = copy.deepcopy(model.state_dict())

    log = {
        'has_negative_curvature': False,
        'n_negative_directions': 0,
        'directions_tried': [],
        'escaped': False,
        'best_direction': None,
        'method': 'deep_escape_v4',
    }

    # Step 1: Full Lanczos decomposition
    print(f"      Computing Lanczos decomposition (k={lanczos_k})...")
    eigenvalues, eigvec_coeffs, Q_list, spectrum_info = lanczos_decomposition(
        model, loss_fn, loader, device, k=lanczos_k, num_batches=5
    )
    log['spectrum'] = spectrum_info
    print(f"      Spectrum: {spectrum_info['n_positive']} pos, {spectrum_info['n_negative']} neg")
    print(f"      Range: [{spectrum_info['min_eigenvalue']:.2f}, {spectrum_info['max_eigenvalue']:.2f}]")

    # Step 2: Extract ALL negative eigenvectors
    neg_eigvecs = extract_all_negative_eigenvectors(
        eigenvalues, eigvec_coeffs, Q_list, device, threshold=-0.1
    )
    log['n_negative_directions'] = len(neg_eigvecs)

    # Free Lanczos basis (already extracted what we need)
    del Q_list, eigvec_coeffs, eigenvalues
    torch.cuda.empty_cache() if device.type == 'cuda' else None

    if len(neg_eigvecs) == 0:
        log['has_negative_curvature'] = False
        log['reason'] = 'no_negative_curvature'
        log['fall_distance'] = 0.0
        print(f"      ⚠️  No negative curvature. Landscape is a BOWL.")
        return log

    log['has_negative_curvature'] = True
    print(f"      ✅ {len(neg_eigvecs)} negative directions found")
    for i, (_, ev) in enumerate(neg_eigvecs[:max_directions]):
        print(f"         Direction {i+1}: λ = {ev:.2f}")

    # Step 3: Try each direction, pick the best
    n_to_try = min(len(neg_eigvecs), max_directions)
    best_escape = None
    best_escape_loss = float('inf')
    best_escape_state = None

    for dir_idx in range(n_to_try):
        direction, eig_val = neg_eigvecs[dir_idx]
        print(f"\n      ─── Direction {dir_idx+1}/{n_to_try}: λ = {eig_val:.2f} ───")

        # Reset model to pre-escape state
        model.load_state_dict(copy.deepcopy(saved_state))

        # Try escaping along this direction
        dir_log = escape_along_direction(
            model, direction, eig_val, loss_fn, loader, device,
            max_steps=max_steps, min_distance=min_distance,
            loss_patience=loss_patience,
        )

        # Also try the OPPOSITE direction (saddle has two sides)
        model.load_state_dict(copy.deepcopy(saved_state))
        neg_direction = [-d for d in direction]
        dir_log_neg = escape_along_direction(
            model, neg_direction, eig_val, loss_fn, loader, device,
            max_steps=max_steps, min_distance=min_distance,
            loss_patience=loss_patience,
        )

        # Pick the better of the two sides
        if dir_log_neg.get('escaped') and not dir_log.get('escaped'):
            dir_log = dir_log_neg
            dir_log['direction_sign'] = 'negative'
        elif dir_log.get('escaped') and dir_log_neg.get('escaped'):
            if dir_log_neg.get('final_loss', float('inf')) < dir_log.get('final_loss', float('inf')):
                dir_log = dir_log_neg
                dir_log['direction_sign'] = 'negative'
            else:
                dir_log['direction_sign'] = 'positive'
        elif not dir_log.get('escaped') and not dir_log_neg.get('escaped'):
            # Neither escaped — pick the one that went further
            if dir_log_neg.get('fall_distance', 0) > dir_log.get('fall_distance', 0):
                dir_log = dir_log_neg
                dir_log['direction_sign'] = 'negative'
            else:
                dir_log['direction_sign'] = 'positive'
        else:
            dir_log['direction_sign'] = 'positive'

        log['directions_tried'].append({
            'eigenvalue': eig_val,
            'escaped': dir_log.get('escaped', False),
            'reason': dir_log.get('reason', '?'),
            'fall_distance': dir_log.get('fall_distance', 0),
            'barrier_height': dir_log.get('barrier_height', 0),
            'final_loss': dir_log.get('final_loss', float('inf')),
            'n_steps': dir_log.get('n_steps', 0),
            'direction_sign': dir_log.get('direction_sign', '?'),
        })

        status = '✅ ESCAPED' if dir_log.get('escaped') else '❌ no escape'
        print(f"      {status}: {dir_log.get('reason', '?')}, "
              f"dist={dir_log.get('fall_distance', 0):.4f}, "
              f"barrier={dir_log.get('barrier_height', 0):.4f}, "
              f"final_loss={dir_log.get('final_loss', 0):.4f}")

        # Track best escape
        if dir_log.get('escaped'):
            final_loss = dir_log.get('final_loss', float('inf'))
            if final_loss < best_escape_loss:
                best_escape = dir_log
                best_escape_loss = final_loss
                best_escape_state = copy.deepcopy(model.state_dict())

    # Step 4: Apply the best escape (or restore if no escape found)
    if best_escape is not None:
        # Reload the best escape state
        # We need to re-walk the best direction to get the right state
        model.load_state_dict(best_escape_state)
        log['escaped'] = True
        log['best_direction'] = {
            'eigenvalue': best_escape.get('eigenvalue', 0),
            'fall_distance': best_escape.get('fall_distance', 0),
            'barrier_height': best_escape.get('barrier_height', 0),
            'final_loss': best_escape_loss,
            'n_steps': best_escape.get('n_steps', 0),
            'reason': best_escape.get('reason', '?'),
        }
        log['fall_distance'] = best_escape.get('fall_distance', 0)
        log['reason'] = best_escape.get('reason', '?')
    else:
        model.load_state_dict(saved_state)
        log['escaped'] = False
        log['fall_distance'] = 0.0
        log['reason'] = 'no_direction_escaped'

    return log


def run_adamw(seed):
    """Fresh AdamW baseline."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_vgg16bn().to(device)
    loss_fn = nn.CrossEntropyLoss()

    t0 = time.time()
    log = train_epochs(model, TOTAL_EPOCHS, tag=f"adamw-s{seed}")
    wall = time.time() - t0

    _, _, _, spectrum_info = lanczos_decomposition(
        model, loss_fn, train_loader, device, k=V4_LANCZOS_K, num_batches=5
    )
    test_acc, _ = evaluate(model, test_loader, device)
    train_acc, _ = evaluate(model, train_loader, device)

    return {
        'best_acc': log['best_acc'], 'test_acc': test_acc, 'train_acc': train_acc,
        'gap': train_acc - test_acc, 'wall_time': wall,
        'sharpness': spectrum_info['max_eigenvalue'],
        'spectrum': spectrum_info,
    }


def run_herd_v4(seed):
    """HERD v4: Warmup → Multi-direction deep escape → Train."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_vgg16bn().to(device)
    loss_fn = nn.CrossEntropyLoss()

    t0 = time.time()

    # Phase 1: Warmup
    print(f"    [v4-s{seed}] Phase 1: Warmup ({V4_WARMUP} epochs)")
    train_epochs(model, V4_WARMUP, tag=f"v4-s{seed}-warmup", verbose=False)
    pre_acc, _ = evaluate(model, test_loader, device)
    print(f"    [v4-s{seed}] Post-warmup: {pre_acc:.2f}%")

    # Phase 2+3: Multi-direction deep escape
    print(f"    [v4-s{seed}] Phase 2+3: Deep escape (loss-barrier, multi-direction)...")
    escape_log = deep_escape_v4(
        model, loss_fn, train_loader, device,
        lanczos_k=V4_LANCZOS_K,
        max_steps=V4_MAX_ESCAPE_STEPS,
        min_distance=V4_MIN_DISTANCE,
        max_directions=V4_MAX_DIRECTIONS,
        loss_patience=V4_LOSS_PATIENCE,
    )

    print(f"    [v4-s{seed}] Escape: {escape_log.get('reason', '?')}, "
          f"neg_curv={'✅' if escape_log['has_negative_curvature'] else '❌'}, "
          f"escaped={'✅' if escape_log['escaped'] else '❌'}, "
          f"dist={escape_log.get('fall_distance', 0):.4f}")

    if escape_log['escaped']:
        post_acc, _ = evaluate(model, test_loader, device)
        print(f"    [v4-s{seed}] Post-escape: {post_acc:.2f}% (was {pre_acc:.2f}%)")

    # Phase 4: Train in (possibly new) basin
    remaining = TOTAL_EPOCHS - V4_WARMUP
    print(f"    [v4-s{seed}] Phase 4: Basin training ({remaining} epochs)")
    log = train_epochs(model, remaining, tag=f"v4-s{seed}-basin")
    wall = time.time() - t0

    # Final measurements
    _, _, _, final_spectrum = lanczos_decomposition(
        model, loss_fn, train_loader, device, k=V4_LANCZOS_K, num_batches=5
    )
    test_acc, _ = evaluate(model, test_loader, device)
    train_acc, _ = evaluate(model, train_loader, device)

    return {
        'best_acc': log['best_acc'], 'test_acc': test_acc, 'train_acc': train_acc,
        'gap': train_acc - test_acc, 'wall_time': wall,
        'sharpness': final_spectrum['max_eigenvalue'],
        'spectrum': final_spectrum,
        'escape_log': escape_log,
    }


print(f"\nHERD v4 (Deep Escape) ready.")
print(f"  Architecture: VGG-16-BN")
print(f"  Escape: loss-barrier + multi-direction + adaptive step")
print(f"  Max directions: {V4_MAX_DIRECTIONS}, Min distance: {V4_MIN_DISTANCE}")
print(f"  Max steps: {V4_MAX_ESCAPE_STEPS}, Loss patience: {V4_LOSS_PATIENCE}")


HERD v4 (Deep Escape) ready.
  Architecture: VGG-16-BN
  Escape: loss-barrier + multi-direction + adaptive step
  Max directions: 3, Min distance: 0.5
  Max steps: 100, Loss patience: 5


In [5]:
# ══════════════════════════════════════════════════════════════
# CELL 5: PART 1 — RUN ADAMW BASELINE (5 seeds)
#
# We have v8 cached results but run fresh anyway for
# exact same-session comparison.
# ══════════════════════════════════════════════════════════════
import os
output_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

print("═" * 70)
print("PART 1: ADAMW BASELINE — VGG-16-BN (5 seeds)")
print("═" * 70)

all_adamw = []
adamw_t0 = time.time()

for i, seed in enumerate(SEEDS):
    print(f"\n{'█' * 70}")
    print(f"  ADAMW SEED {i+1}/{len(SEEDS)}: {seed}")
    print(f"  v8 reference: {CACHED_ADAMW_V8[seed]['best_acc']:.2f}%")
    print(f"{'█' * 70}")

    result = run_adamw(seed)
    all_adamw.append(result)

    print(f"\n  Seed {seed}: best_acc={result['best_acc']:.2f}% "
          f"(v8 ref: {CACHED_ADAMW_V8[seed]['best_acc']:.2f}%)")

    elapsed = time.time() - adamw_t0
    est_remain = elapsed / (i + 1) * (len(SEEDS) - i - 1)
    print(f"  Elapsed: {elapsed/3600:.1f}h | Est. remaining: {est_remain/3600:.1f}h")

adamw_time = time.time() - adamw_t0
adamw_accs = np.array([r['best_acc'] for r in all_adamw])
print(f"\n{'═' * 70}")
print(f"AdamW DONE. Runtime: {adamw_time/3600:.2f}h")
print(f"AdamW mean: {adamw_accs.mean():.2f}% ± {adamw_accs.std():.2f}%")
print(f"{'═' * 70}")

══════════════════════════════════════════════════════════════════════
PART 1: ADAMW BASELINE — VGG-16-BN (5 seeds)
══════════════════════════════════════════════════════════════════════

██████████████████████████████████████████████████████████████████████
  ADAMW SEED 1/5: 42
  v8 reference: 91.35%
██████████████████████████████████████████████████████████████████████
VGG-16-BN (CIFAR): 15,253,578 parameters
    [adamw-s42] Ep  10/100 | TestAcc: 66.57% | Best: 66.57%
    [adamw-s42] Ep  20/100 | TestAcc: 78.24% | Best: 78.24%
    [adamw-s42] Ep  30/100 | TestAcc: 83.12% | Best: 83.12%
    [adamw-s42] Ep  40/100 | TestAcc: 87.31% | Best: 87.31%
    [adamw-s42] Ep  50/100 | TestAcc: 88.72% | Best: 88.72%
    [adamw-s42] Ep  60/100 | TestAcc: 90.42% | Best: 90.42%
    [adamw-s42] Ep  70/100 | TestAcc: 90.62% | Best: 90.62%
    [adamw-s42] Ep  80/100 | TestAcc: 91.56% | Best: 91.56%
    [adamw-s42] Ep  90/100 | TestAcc: 91.64% | Best: 91.64%
    [adamw-s42] Ep 100/100 | TestAcc: 91.57% 

In [6]:
# ══════════════════════════════════════════════════════════════
# CELL 6: PART 2 — RUN HERD v4 (5 seeds)
# ══════════════════════════════════════════════════════════════
print("═" * 70)
print("PART 2: HERD v4 DEEP ESCAPE — VGG-16-BN (5 seeds)")
print("═" * 70)

all_v4 = []
v4_t0 = time.time()

for i, seed in enumerate(SEEDS):
    print(f"\n{'█' * 70}")
    print(f"  v4 SEED {i+1}/{len(SEEDS)}: {seed}")
    print(f"  AdamW ref: {all_adamw[i]['best_acc']:.2f}%")
    print(f"{'█' * 70}")

    result = run_herd_v4(seed)
    all_v4.append(result)

    esc = result['escape_log']
    print(f"\n  ┌─ Seed {seed} ────────────────────────────────────────┐")
    print(f"  │ AdamW:    {all_adamw[i]['best_acc']:.2f}%")
    print(f"  │ v4:       {result['best_acc']:.2f}%  "
          f"neg_curv={'✅' if esc['has_negative_curvature'] else '❌'} "
          f"escaped={'✅' if esc['escaped'] else '❌'}")
    print(f"  │ Δ(v4-AdamW): {result['best_acc']-all_adamw[i]['best_acc']:+.2f}%")
    if esc.get('best_direction'):
        bd = esc['best_direction']
        print(f"  │ Best direction: λ={bd.get('eigenvalue', 0):.2f}, "
              f"barrier={bd.get('barrier_height', 0):.4f}, "
              f"dist={bd.get('fall_distance', 0):.4f}")
    print(f"  │ Directions tried: {len(esc.get('directions_tried', []))}")
    print(f"  └────────────────────────────────────────────────────┘")

    elapsed = time.time() - v4_t0
    est_remain = elapsed / (i + 1) * (len(SEEDS) - i - 1)
    print(f"  Elapsed: {elapsed/3600:.1f}h | Est. remaining: {est_remain/3600:.1f}h")

v4_time = time.time() - v4_t0
v4_accs = np.array([r['best_acc'] for r in all_v4])
print(f"\n{'═' * 70}")
print(f"v4 DONE. Runtime: {v4_time/3600:.2f}h")
print(f"v4 mean:    {v4_accs.mean():.2f}% ± {v4_accs.std():.2f}%")
print(f"AdamW mean: {adamw_accs.mean():.2f}% ± {adamw_accs.std():.2f}%")
print(f"Δ mean: {(v4_accs - adamw_accs).mean():+.2f}%")
print(f"{'═' * 70}")

══════════════════════════════════════════════════════════════════════
PART 2: HERD v4 DEEP ESCAPE — VGG-16-BN (5 seeds)
══════════════════════════════════════════════════════════════════════

██████████████████████████████████████████████████████████████████████
  v4 SEED 1/5: 42
  AdamW ref: 91.64%
██████████████████████████████████████████████████████████████████████
VGG-16-BN (CIFAR): 15,253,578 parameters
    [v4-s42] Phase 1: Warmup (10 epochs)
    [v4-s42] Post-warmup: 61.09%
    [v4-s42] Phase 2+3: Deep escape (loss-barrier, multi-direction)...
      Computing Lanczos decomposition (k=20)...
      Spectrum: 13 pos, 6 neg
      Range: [-29.89, 285.17]
      ✅ 6 negative directions found
         Direction 1: λ = -29.89
         Direction 2: λ = -23.04
         Direction 3: λ = -6.86

      ─── Direction 1/3: λ = -29.89 ───
          Step 20: loss=5.6590 (start=1.4066, max=5.6590), dist=3.6582, barrier=✅, below_start=0
          Step 40: loss=10.8429 (start=1.4066, max=10.8429), 

In [7]:
# ══════════════════════════════════════════════════════════════
# CELL 7: ANALYSIS — Hypothesis Tests
# ══════════════════════════════════════════════════════════════
alpha = 0.05

adamw_accs   = np.array([r['best_acc'] for r in all_adamw])
adamw_sharps = np.array([r['sharpness'] for r in all_adamw])
v4_accs      = np.array([r['best_acc'] for r in all_v4])
v4_sharps    = np.array([r['sharpness'] for r in all_v4])

print("═" * 70)
print("HYPOTHESIS TESTS")
print("═" * 70)

# ── H1: Loss-barrier escape detects genuine basin crossings ──
print("\n─── H1: Loss-barrier escape detects genuine basin crossings ───")
n_barrier_crossed = sum(
    1 for r in all_v4
    if any(d.get('escaped') and 'barrier' in d.get('reason', '')
           for d in r['escape_log'].get('directions_tried', []))
)
n_escaped = sum(1 for r in all_v4 if r['escape_log']['escaped'])
n_has_neg = sum(1 for r in all_v4 if r['escape_log']['has_negative_curvature'])
print(f"  Seeds with negative curvature: {n_has_neg}/{len(SEEDS)}")
print(f"  Seeds with barrier crossing: {n_barrier_crossed}/{len(SEEDS)}")
print(f"  Seeds that escaped: {n_escaped}/{len(SEEDS)}")
h1_supported = n_barrier_crossed >= 3
print(f"  {'✅ H1 SUPPORTED' if h1_supported else '❌ H1 REJECTED'}")

# ── H2: Escape distance >> v8 ───────────────────────────────
print("\n─── H2: Escape distance significantly larger than v8 ───")
v8_distances = [0.05, 0.05, 0.05, 0.20, 0.40]  # from v8 results
v4_distances = [r['escape_log'].get('fall_distance', 0) for r in all_v4]
print(f"  v8 distances: {v8_distances}")
print(f"  v4 distances: {[f'{d:.4f}' for d in v4_distances]}")
v4_avg_dist = np.mean(v4_distances)
v8_avg_dist = np.mean(v8_distances)
h2_supported = v4_avg_dist > v8_avg_dist * 2  # at least 2x further
print(f"  v4 avg: {v4_avg_dist:.4f}, v8 avg: {v8_avg_dist:.4f}")
print(f"  {'✅ H2 SUPPORTED' if h2_supported else '❌ H2 REJECTED'}")

# ── H3: Post-escape basin has different spectrum ─────────────
print("\n─── H3: Post-escape basin has different loss landscape ───")
# Compare pre-escape spectrum (from escape_log) vs final spectrum
n_spectrum_changed = 0
for i, r in enumerate(all_v4):
    esc = r['escape_log']
    if esc.get('spectrum') and r.get('spectrum'):
        pre_max = esc['spectrum']['max_eigenvalue']
        post_max = r['spectrum']['max_eigenvalue']
        change = abs(post_max - pre_max) / (abs(pre_max) + 1e-10)
        changed = change > 0.3  # >30% change in top eigenvalue
        if changed:
            n_spectrum_changed += 1
        print(f"  Seed {SEEDS[i]}: pre_λ_max={pre_max:.1f}, post_λ_max={post_max:.1f}, "
              f"change={change:.1%} {'✅' if changed else ''}")
h3_supported = n_spectrum_changed >= 3
print(f"  {'✅ H3 SUPPORTED' if h3_supported else '❌ H3 REJECTED'}")

# ── H4: v4 beats AdamW ──────────────────────────────────────
print("\n─── H4: v4 accuracy vs AdamW ───")

def paired_test(name, base, test_vals, direction):
    diff = test_vals - base
    t_stat, t_pval = stats.ttest_rel(test_vals, base)
    try:
        w_stat, w_pval = stats.wilcoxon(diff, alternative='greater' if direction == 'higher' else 'less')
    except:
        w_stat, w_pval = 0, 1.0
    d = diff.mean() / (diff.std(ddof=1) + 1e-10)
    sig = t_pval < alpha
    correct = (direction == 'higher' and diff.mean() > 0) or (direction == 'lower' and diff.mean() < 0)
    if sig and correct:
        verdict = "✅ SUPPORTED"
    elif sig:
        verdict = "❌ REJECTED (wrong direction)"
    else:
        verdict = "❌ REJECTED (not significant)"
    print(f"  {name}: Δ={diff.mean():+.3f}% ± {diff.std():.3f}, "
          f"t={t_stat:.3f}, p_t={t_pval:.4f}, p_w={w_pval:.4f}, d={d:.3f} → {verdict}")
    return {'t_pval': t_pval, 'w_pval': w_pval, 'd': d, 'verdict': verdict}

h4_test = paired_test("v4 vs AdamW (acc)", adamw_accs, v4_accs, 'higher')
h4_supported = "SUPPORTED" in h4_test['verdict']

# ── H5: Flatter minima ──────────────────────────────────────
print("\n─── H5: v4 sharpness vs AdamW ───")
h5_test = paired_test("v4 vs AdamW (sharpness)", adamw_sharps, v4_sharps, 'lower')
h5_supported = "SUPPORTED" in h5_test['verdict']

# ══════════════════════════════════════════════════════════════
# FINAL VERDICT
# ══════════════════════════════════════════════════════════════
print(f"\n\n{'█' * 70}")
print("█" + " " * 20 + "FINAL VERDICT" + " " * 35 + "█")
print(f"{'█' * 70}")

all_h = {
    'H1 (loss-barrier detection)': h1_supported,
    'H2 (deeper escape)': h2_supported,
    'H3 (different spectrum)': h3_supported,
    'H4 (v4 beats AdamW)': h4_supported,
    'H5 (flatter minima)': h5_supported,
}
for name, sup in all_h.items():
    print(f"\n  {'✅' if sup else '❌'} {name}")

n_supported = sum(all_h.values())
print(f"\n{'─' * 70}")
print(f"  SUPPORTED: {n_supported}/5")
print(f"{'─' * 70}")

# ── Comparison table ─────────────────────────────────────────
print(f"\n\n{'═' * 70}")
print("ACCURACY COMPARISON")
print(f"{'═' * 70}")
print(f"\n{'Seed':<8} {'AdamW':>8} {'v4':>8} {'Δ':>8} {'Escaped':>8} {'Dist':>8} {'Barrier':>9}")
print("─" * 60)
for i, s in enumerate(SEEDS):
    esc = all_v4[i]['escape_log']
    bd = esc.get('best_direction', {})
    print(f"{s:<8} {all_adamw[i]['best_acc']:>7.2f}% {all_v4[i]['best_acc']:>7.2f}% "
          f"{all_v4[i]['best_acc']-all_adamw[i]['best_acc']:>+7.2f}% "
          f"{'✅' if esc['escaped'] else '❌':>7} "
          f"{esc.get('fall_distance', 0):>7.3f} "
          f"{bd.get('barrier_height', 0):>8.4f}")
print("─" * 60)
print(f"{'Mean':<8} {adamw_accs.mean():>7.2f}% {v4_accs.mean():>7.2f}% "
      f"{(v4_accs-adamw_accs).mean():>+7.2f}%")

# ── The big picture ──────────────────────────────────────────
print(f"\n\n{'═' * 70}")
print("THE BIG PICTURE")
print(f"{'═' * 70}")

if h1_supported and h2_supported:
    if h4_supported:
        print("""
  ⚡ DEEP ESCAPE WORKS! Loss-barrier detection finds genuine basin crossings.

  Key findings:
  • Loss-barrier criterion is robust (loss ↑ then ↓ = real saddle crossing)
  • Escape distance is much larger than v8 (actual basin change, not jitter)
  • Different basin trains to HIGHER accuracy

  HERD v4 successfully navigates the loss landscape!
""")
    else:
        print("""
  🔍 Deep escape MECHANICALLY works — we cross real barriers and
  land in genuinely different basins. But the new basins don't train
  to higher accuracy. All basins may be equally good.

  This suggests the landscape has multi-basin structure (HERD's premise
  is correct) but the basins are equivalent in quality.
""")
else:
    print("""
  The loss-barrier criterion may be too strict (no barriers found)
  or the landscape doesn't have the expected barrier structure.
  The saddle directions exist (v8 confirmed) but may not connect
  to meaningfully separated basins.
""")

══════════════════════════════════════════════════════════════════════
HYPOTHESIS TESTS
══════════════════════════════════════════════════════════════════════

─── H1: Loss-barrier escape detects genuine basin crossings ───
  Seeds with negative curvature: 5/5
  Seeds with barrier crossing: 0/5
  Seeds that escaped: 0/5
  ❌ H1 REJECTED

─── H2: Escape distance significantly larger than v8 ───
  v8 distances: [0.05, 0.05, 0.05, 0.2, 0.4]
  v4 distances: ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000']
  v4 avg: 0.0000, v8 avg: 0.1500
  ❌ H2 REJECTED

─── H3: Post-escape basin has different loss landscape ───
  Seed 42: pre_λ_max=285.2, post_λ_max=24.8, change=91.3% ✅
  Seed 123: pre_λ_max=346.7, post_λ_max=32.3, change=90.7% ✅
  Seed 777: pre_λ_max=209.8, post_λ_max=18.8, change=91.0% ✅
  Seed 2024: pre_λ_max=117.9, post_λ_max=32.4, change=72.5% ✅
  Seed 31415: pre_λ_max=204.9, post_λ_max=42.6, change=79.2% ✅
  ✅ H3 SUPPORTED

─── H4: v4 accuracy vs AdamW ───
  v4 vs AdamW (acc): Δ=+0

AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8: VISUALIZATION
# ══════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 16))

# ── PLOT 1: Escape loss trajectories ─────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
for i, s in enumerate(SEEDS):
    esc = all_v4[i]['escape_log']
    # Find the best direction's steps
    best_dir = esc.get('best_direction', {})
    # Plot from directions_tried
    for j, d in enumerate(esc.get('directions_tried', [])):
        # We don't store full step histories in directions_tried summary
        # Just plot markers for key metrics
        pass

    # Plot escape distances as bars
    dist = esc.get('fall_distance', 0)
    escaped = esc['escaped']
    color = '#27ae60' if escaped else '#e74c3c'
    ax1.bar(i, dist, color=color, alpha=0.7, label='escaped' if i == 0 and escaped else '')

# v8 reference distances
v8_dists = [0.05, 0.05, 0.05, 0.20, 0.40]
ax1.scatter(range(len(SEEDS)), v8_dists, marker='x', color='gray', s=100, zorder=5,
           label='v8 distance (too shallow)')
ax1.set_xticks(range(len(SEEDS)))
ax1.set_xticklabels([f'Seed {s}' for s in SEEDS], fontsize=9)
ax1.set_ylabel('Escape Distance (parameter space)')
ax1.set_title('v4 Deep Escape Distance vs v8 Shallow Escape')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# ── PLOT 2: Accuracy comparison (v8 AdamW, v8 v3f, v9 AdamW, v9 v4) ─
ax2 = fig.add_subplot(2, 2, 2)
v8_adamw_accs = np.array([CACHED_ADAMW_V8[s]['best_acc'] for s in SEEDS])
v8_v3f_accs = np.array([92.12, 91.92, 91.82, 92.58, 91.99])  # from v8 results

data_box = [adamw_accs, v4_accs]
labels_box = ['AdamW', 'HERD v4\n(deep escape)']
colors_box = ['#3498db', '#9b59b6']

bp = ax2.boxplot(data_box, labels=labels_box, patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
for i, (accs, color) in enumerate(zip(data_box, colors_box)):
    jitter = np.random.RandomState(42).uniform(-0.08, 0.08, len(accs))
    ax2.scatter([i+1]*len(accs) + jitter, accs, color=color, s=60, zorder=5,
               edgecolors='black', linewidth=0.5)
ax2.set_ylabel('Best Test Accuracy (%)')
ax2.set_title('VGG-16-BN: AdamW vs HERD v4')
ax2.grid(True, alpha=0.3, axis='y')
for i, accs in enumerate(data_box):
    ax2.annotate(f'{accs.mean():.2f}%', xy=(i+1, accs.mean()),
                 xytext=(i+1+0.35, accs.mean()),
                 fontsize=10, fontweight='bold', color=colors_box[i])

# ── PLOT 3: Barrier heights per direction per seed ───────────
ax3 = fig.add_subplot(2, 2, 3)
for i, s in enumerate(SEEDS):
    dirs = all_v4[i]['escape_log'].get('directions_tried', [])
    for j, d in enumerate(dirs):
        marker = 'o' if d.get('escaped') else 'x'
        color = plt.cm.Set2(i / len(SEEDS))
        ax3.scatter(abs(d.get('eigenvalue', 0)), d.get('barrier_height', 0),
                   marker=marker, s=80, color=color, edgecolors='black', linewidth=0.5,
                   label=f'Seed {s}' if j == 0 else '')
ax3.set_xlabel('|λ| (curvature strength)')
ax3.set_ylabel('Barrier height (loss)')
ax3.set_title('Curvature vs Barrier Height per Direction')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# ── PLOT 4: Sharpness comparison ─────────────────────────────
ax4 = fig.add_subplot(2, 2, 4)
x = np.arange(len(SEEDS))
width = 0.35
ax4.bar(x - width/2, adamw_sharps, width, label='AdamW', color='#3498db', alpha=0.7)
ax4.bar(x + width/2, v4_sharps, width, label='HERD v4', color='#9b59b6', alpha=0.7)
ax4.set_xticks(x)
ax4.set_xticklabels([f'Seed {s}' for s in SEEDS], fontsize=9)
ax4.set_ylabel('Sharpness (λ_max)')
ax4.set_title('Final Sharpness: AdamW vs v4')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('HERD v9: Deep Escape (Loss-Barrier + Multi-Direction)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'herd_v9_deep_escape.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Saved: {os.path.join(output_dir, 'herd_v9_deep_escape.png')}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9: JSON EXPORT
# ══════════════════════════════════════════════════════════════
import json

results = {
    'experiment': 'HERD v9 — Deep Escape: Loss-Barrier + Multi-Direction + Adaptive Step',
    'model': 'VGG-16-BN (CIFAR-10 adapted, no skip connections)',
    'dataset': 'CIFAR-10',
    'v8_postmortem': (
        'v8 escape was too shallow: cosine threshold triggered after 1 step (dist=0.05) '
        'because gradient noise flips cosine trivially. Model never left its basin.'
    ),
    'v9_fixes': {
        'loss_barrier_criterion': 'Require loss to go UP (cross barrier) then DOWN (new basin)',
        'adaptive_step_size': 'lr = 1/sqrt(|lambda|), clamped to [0.01, 0.5]',
        'multi_direction': f'Try top-{V4_MAX_DIRECTIONS} most negative eigenvectors + both signs',
        'min_distance': f'Must move at least {V4_MIN_DISTANCE} in parameter space',
    },
    'seeds': SEEDS,
    'config': {
        'warmup_epochs': V4_WARMUP,
        'total_epochs': TOTAL_EPOCHS,
        'lanczos_k': V4_LANCZOS_K,
        'max_escape_steps': V4_MAX_ESCAPE_STEPS,
        'min_distance': V4_MIN_DISTANCE,
        'max_directions': V4_MAX_DIRECTIONS,
        'loss_patience': V4_LOSS_PATIENCE,
    },
    'hypotheses': {
        'H1_loss_barrier': {
            'description': 'Loss-barrier escape detects genuine basin crossings',
            'supported': h1_supported,
            'evidence': {'n_barrier_crossed': n_barrier_crossed, 'n_escaped': n_escaped}
        },
        'H2_deeper_escape': {
            'description': 'Escape distance much larger than v8',
            'supported': h2_supported,
            'evidence': {'v4_avg_dist': v4_avg_dist, 'v8_avg_dist': v8_avg_dist}
        },
        'H3_different_spectrum': {
            'description': 'Post-escape basin has different loss landscape',
            'supported': h3_supported,
            'evidence': {'n_spectrum_changed': n_spectrum_changed}
        },
        'H4_v4_beats_adamw': {
            'description': 'v4 achieves higher test accuracy than AdamW',
            'supported': h4_supported,
            'evidence': h4_test
        },
        'H5_flatter_minima': {
            'description': 'v4 finds flatter minima',
            'supported': h5_supported,
            'evidence': h5_test
        },
    },
    'accuracy': {
        'adamw': {
            'per_seed': {str(SEEDS[i]): all_adamw[i]['best_acc'] for i in range(len(SEEDS))},
            'mean': float(adamw_accs.mean()),
            'std': float(adamw_accs.std()),
            'sharpness_mean': float(adamw_sharps.mean()),
        },
        'herd_v4': {
            'per_seed': {str(SEEDS[i]): all_v4[i]['best_acc'] for i in range(len(SEEDS))},
            'mean': float(v4_accs.mean()),
            'std': float(v4_accs.std()),
            'sharpness_mean': float(v4_sharps.mean()),
        },
    },
    'v4_escape_details': [
        {
            'seed': SEEDS[i],
            'best_acc': all_v4[i]['best_acc'],
            'sharpness': all_v4[i]['sharpness'],
            'gap': all_v4[i]['gap'],
            'escape_summary': {
                'escaped': all_v4[i]['escape_log']['escaped'],
                'has_negative_curvature': all_v4[i]['escape_log']['has_negative_curvature'],
                'n_directions_tried': len(all_v4[i]['escape_log'].get('directions_tried', [])),
                'fall_distance': all_v4[i]['escape_log'].get('fall_distance', 0),
                'reason': all_v4[i]['escape_log'].get('reason', '?'),
                'best_direction': all_v4[i]['escape_log'].get('best_direction', {}),
                'directions_tried': all_v4[i]['escape_log'].get('directions_tried', []),
            },
        }
        for i in range(len(SEEDS))
    ],
    'summary': {
        'hypotheses_supported': n_supported,
        'hypotheses_total': 5,
        'conclusion': (
            'Deep escape with loss-barrier detection works and improves accuracy'
            if (h1_supported and h4_supported)
            else 'Deep escape crosses real barriers but basins have similar quality'
            if (h1_supported and h2_supported and not h4_supported)
            else 'Loss-barrier criterion too strict or no real barriers exist'
            if not h1_supported
            else 'Mixed results'
        ),
    },
}

out_path = os.path.join(output_dir, 'herd_v9_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print("═" * 70)
print(f"📦 SAVED: {out_path}")
print("═" * 70)
print(f"\nConclusion: {results['summary']['conclusion']}")
print(f"Hypotheses supported: {n_supported}/5")
print(f"\nAccuracy:")
print(f"  AdamW: {adamw_accs.mean():.2f}% ± {adamw_accs.std():.2f}%")
print(f"  v4:    {v4_accs.mean():.2f}% ± {v4_accs.std():.2f}%")
print(f"  Δ:     {(v4_accs-adamw_accs).mean():+.2f}%")
print(f"\n🏁 Experiment complete.")